# Automatisation de telechargement 

## BULLETTINS_DE_COTE

In [1]:
import os
import requests
import pandas as pd # Utile pour générer les dates facilement
from datetime import datetime
import urllib3
import time

# Désactiver les alertes SSL
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# 1. Configuration
DOSSIER_SAVE = "BULLETINS_COTE_2020_2026"
BASE_URL_PATTERN = "https://media.casablanca-bourse.com/sites/default/files/es-auto-upload/fr/BCFR_{date_string}.pdf"

if not os.path.exists(DOSSIER_SAVE):
    os.makedirs(DOSSIER_SAVE)

def telecharger_bulletins():
    # 2. Générer la liste de toutes les dates du 01/01/2020 à aujourd'hui
    # Note : La bourse est fermée le week-end, mais on va tester toutes les dates au cas où
    start_date = "2020-01-01"
    end_date = datetime.now().strftime("%Y-%m-%d")
    dates = pd.date_range(start=start_date, end=end_date)

    print(f"Début du scan pour {len(dates)} dates potentielles...")

    for date in dates:
        # Formater la date en AAAAMMJJ (ex: 20260123)
        date_str = date.strftime("%Y%m%d")
        url = BASE_URL_PATTERN.format(date_string=date_str)
        nom_fichier = f"BCFR_{date_str}.pdf"
        chemin_final = os.path.join(DOSSIER_SAVE, nom_fichier)

        # Skip si déjà téléchargé
        if os.path.exists(chemin_final):
            continue

        try:
            # On utilise une requête HEAD d'abord pour vérifier si le fichier existe 
            # sans le télécharger (plus rapide et poli pour le serveur)
            check = requests.head(url, verify=False, timeout=5)
            
            if check.status_code == 200:
                print(f"Trouvé ! Téléchargement de : {nom_fichier}")
                r = requests.get(url, verify=False)
                with open(chemin_final, 'wb') as f:
                    f.write(r.content)
                time.sleep(0.1) # Petite pause
            else:
                # C'est probablement un week-end ou un jour férié
                pass

        except Exception as e:
            print(f"Erreur pour la date {date_str}: {e}")

if __name__ == "__main__":
    telecharger_bulletins()

Début du scan pour 2225 dates potentielles...
Erreur pour la date 20211105: HTTPSConnectionPool(host='media.casablanca-bourse.com', port=443): Read timed out. (read timeout=5)
Trouvé ! Téléchargement de : BCFR_20230403.pdf
Trouvé ! Téléchargement de : BCFR_20230407.pdf
Trouvé ! Téléchargement de : BCFR_20230410.pdf
Trouvé ! Téléchargement de : BCFR_20230411.pdf
Trouvé ! Téléchargement de : BCFR_20230412.pdf
Trouvé ! Téléchargement de : BCFR_20230413.pdf
Trouvé ! Téléchargement de : BCFR_20230414.pdf
Trouvé ! Téléchargement de : BCFR_20230417.pdf
Trouvé ! Téléchargement de : BCFR_20230418.pdf
Trouvé ! Téléchargement de : BCFR_20230419.pdf
Trouvé ! Téléchargement de : BCFR_20230420.pdf
Trouvé ! Téléchargement de : BCFR_20230421.pdf
Trouvé ! Téléchargement de : BCFR_20230425.pdf
Trouvé ! Téléchargement de : BCFR_20230426.pdf
Trouvé ! Téléchargement de : BCFR_20230427.pdf
Trouvé ! Téléchargement de : BCFR_20230428.pdf
Trouvé ! Téléchargement de : BCFR_20230502.pdf
Trouvé ! Téléchargement d

## RESUMES_BOURSE

In [ ]:
import os
import requests
import pandas as pd
from datetime import datetime
import urllib3
import time

# Désactivation des alertes SSL
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- CONFIGURATION ---
DOSSIER_RESUMES = "DATA_BOURSE_RESUMES"
# Nouveau pattern basé sur votre lien
URL_PATTERN_RS = "https://media.casablanca-bourse.com/sites/default/files/es-auto-upload/fr/resume_seance_{date_str}.pdf"

if not os.path.exists(DOSSIER_RESUMES):
    os.makedirs(DOSSIER_RESUMES)

def telecharger_resumes():
    # Génération des dates de début 2020 à aujourd'hui
    date_debut = "2020-01-01"
    date_fin = datetime.now().strftime("%Y-%m-%d")
    toutes_les_dates = pd.date_range(start=date_debut, end=date_fin)

    print(f" Lancement de la récupération des Résumés de Séance...")
    
    succes = 0
    deja_presents = 0

    for d in toutes_les_dates:
        # On ne traite que les jours de semaine (Lundi=0, Vendredi=4)
        # La bourse est fermée le week-end (5 et 6)
        if d.weekday() > 4:
            continue

        str_d = d.strftime("%Y%m%d")
        url = URL_PATTERN_RS.format(date_str=str_d)
        nom_fichier = f"resume_seance_{str_d}.pdf"
        chemin_fichier = os.path.join(DOSSIER_RESUMES, nom_fichier)

        if os.path.exists(chemin_fichier):
            deja_presents += 1
            continue

        try:
            # Vérification de l'existence sur le serveur
            check = requests.head(url, verify=False, timeout=5)
            
            if check.status_code == 200:
                print(f" Trouvé et téléchargé : {nom_fichier}")
                r = requests.get(url, verify=False)
                with open(chemin_fichier, 'wb') as f:
                    f.write(r.content)
                succes += 1
                time.sleep(0.1) # Pause pour la courtoisie serveur
            
        except Exception as e:
            print(f" Erreur pour la date {str_d} : {e}")

    print(f"\n Mission terminée !")
    print(f" Nouveaux résumés récupérés : {succes}")
    print(f" Total dans le dossier : {len(os.listdir(DOSSIER_RESUMES))}")

if __name__ == "__main__":
    telecharger_resumes()

🚀 Lancement de la récupération des Résumés de Séance...
📥 Trouvé et téléchargé : resume_seance_20230407.pdf
📥 Trouvé et téléchargé : resume_seance_20230410.pdf
📥 Trouvé et téléchargé : resume_seance_20230411.pdf
📥 Trouvé et téléchargé : resume_seance_20230412.pdf
📥 Trouvé et téléchargé : resume_seance_20230413.pdf
📥 Trouvé et téléchargé : resume_seance_20230414.pdf
📥 Trouvé et téléchargé : resume_seance_20230417.pdf
📥 Trouvé et téléchargé : resume_seance_20230418.pdf
📥 Trouvé et téléchargé : resume_seance_20230419.pdf
📥 Trouvé et téléchargé : resume_seance_20230420.pdf
📥 Trouvé et téléchargé : resume_seance_20230421.pdf
📥 Trouvé et téléchargé : resume_seance_20230425.pdf
📥 Trouvé et téléchargé : resume_seance_20230426.pdf
📥 Trouvé et téléchargé : resume_seance_20230427.pdf
📥 Trouvé et téléchargé : resume_seance_20230428.pdf
📥 Trouvé et téléchargé : resume_seance_20230502.pdf
📥 Trouvé et téléchargé : resume_seance_20230503.pdf
📥 Trouvé et téléchargé : resume_seance_20230504.pdf
📥 Trouvé